# Bayesian Classification

In [1]:
import scipy.io
import numpy as np
import sklearn
from sklearn.feature_extraction import image
import matplotlib.pyplot as plt
from helper_functions import *

## Dataset - Data

In [2]:
data = scipy.io.loadmat('Data/data.mat')
faces = np.array(data['face'])
#number of classes 
M = 200
N = 600 # 3 per class

   ### Split Data

In [3]:

#select images 1 and 2 from each subject for training
faces_flat = faces.reshape(-1,faces.shape[2])
#print(faces_flat.shape)
train_idx =[]
test_idx = []
for n in range(M):
    idx = [3*n+1, 3*n+2]
    train_idx = train_idx + idx
    test_idx = test_idx + [3*n]
#print(test_idx)
training_faces = faces_flat[:,train_idx]
test_faces = faces_flat[:,test_idx]
#print(training_faces.shape)
#print(test_faces.shape)



### Perform PCA

In [4]:
#reduce dimensions from 504 to 100 (only 400 obsercations, sigma_w will be non invertible for 504).
A = PCA(training_faces, 50)
X_PCA = A @ training_faces
X_test_PCA = A @ test_faces
print(test_faces.shape)


(504, 200)


In [5]:
# test = A[0,:].T
# test = test.reshape(24,21)
# plot = plt.imshow(test)
# plt.show()

### Perform MDA

In [6]:
#reduce dimensions from 100 to 50
A,class_means,sigma_w = MDA(X_PCA, 30, M)
X_MDA = A @ X_PCA
X_test_MDA = A @ X_test_PCA
print(X_MDA.shape)

(30, 400)


### Parametric Estimation

In [7]:
#after MDA, assumtion is all classes share covariance A@sigma_w@A.T, with means A@class_means
print(class_means.shape)
print(A.shape)
class_means = A @ class_means
sigma_w = A @ sigma_w @ A.T
print(sigma_w.shape)

(50, 200)
(30, 50)
(30, 30)


### Bayes Classifier

In [8]:
# For shared covariance and equal priors, the classifier is the minimum Mahalanobis distance classifier
# training error
N_train = 400
sigma_inv = np.linalg.inv(sigma_w)
#find mahalanobis distances for training data
distances = np.zeros((N_train,M))
for i in range(M):
    delta = X_MDA - class_means[:,i].reshape(-1,1)
    #vectorized
    sigma_delta = sigma_inv @ delta
    distances[:,i] = np.sum(delta * sigma_delta, axis = 0)
classified = np.argmin(distances, axis = 1)
y_values = [i for i in range(200) for _ in range(2)]

error = 1/len(y_values)*sum(a!=b for (a,b) in zip(classified,y_values))
print(error)

N_test = 200
#find mahalanobis distances for training data
distances = np.zeros((N_test,M))
for i in range(M):
    delta = X_test_MDA - class_means[:,i].reshape(-1,1)
    #vectorized
    sigma_delta = sigma_inv @ delta
    distances[:,i] = np.sum(delta * sigma_delta, axis = 0)
classified = np.argmin(distances, axis = 1)
y_values = [i for i in range(200)]

error = 1/len(y_values)*sum(a!=b for (a,b) in zip(classified,y_values))
print(error)


0.0
0.215


## Dataset - Pose

In [9]:
data = scipy.io.loadmat('Data/pose.mat')
faces = np.array(data['pose'])
M = 68

In [10]:
print(faces[:,:,0,0].shape)

(48, 40)


### Split data

In [11]:
faces_flat = faces.reshape(-1, faces.shape[2], faces.shape[3])  # flatten images

train_idx = list(range(0,9))
test_idx = list(range(9,13))

# Vectorized selection
# training_faces: (pixels, 9 images * 68 subjects) first 9 images of subj 0, and so on
training_faces = faces_flat[:, train_idx, :].swapaxes(1, 2).reshape(faces_flat.shape[0], -1)

# test_faces: (pixels, 4 images * 68 subjects)
test_faces = faces_flat[:, test_idx, :].swapaxes(1, 2).reshape(faces_flat.shape[0], -1)

print("Training faces shape:", training_faces.shape)
print("Test faces shape:", test_faces.shape)

Training faces shape: (1920, 612)
Test faces shape: (1920, 272)


### PCA and MDA

In [12]:
#PCA - reduce dimensions from 504 to 100 (only 400 obsercations, sigma_w will be non invertible for 504).
A = PCA(training_faces, 70)
X_PCA = A @ training_faces
X_test_PCA = A @ test_faces
print(X_PCA.shape)

#MDA - reduce dimensions from 100 to 50
A,class_means,sigma_w = MDA(X_PCA, 40, M)
A = np.real(A)
X_MDA = A @ X_PCA
X_test_MDA = A @ X_test_PCA
print(X_MDA.shape)

(70, 612)
(40, 612)


### Estimation

In [13]:
#after MDA, assumtion is all classes share covariance A@sigma_w@A.T, with means A@class_means
print(class_means.shape)
class_means = A @ class_means
sigma_w = A @ sigma_w @ A.T
print(sigma_w.shape)

(70, 68)
(40, 40)


### Bayes Classifier

In [14]:
# For shared covariance and equal priors, the classifier is the minimum Mahalanobis distance classifier
# training error
N_train = 612
sigma_inv = np.linalg.inv(sigma_w)
#find mahalanobis distances for training data
distances = np.zeros((N_train,M))
for i in range(M):
    delta = X_MDA - class_means[:,i].reshape(-1,1)
    #vectorized
    sigma_delta = sigma_inv @ delta
    distances[:,i] = np.sum(delta * sigma_delta, axis = 0)
classified = np.argmin(distances, axis = 1)
y_values = [i for i in range(68) for _ in range(9)]

error = 1/len(y_values)*sum(a!=b for (a,b) in zip(classified,y_values))
print(error)

N_test = 272
#find mahalanobis distances for training data
distances = np.zeros((N_test,M))
for i in range(M):
    delta = X_test_MDA - class_means[:,i].reshape(-1,1)
    #vectorized
    sigma_delta = sigma_inv @ delta
    distances[:,i] = np.sum(delta * sigma_delta, axis = 0)
classified = np.argmin(distances, axis = 1)
y_values = [i for i in range(68) for _ in range(4)]

error = 1/len(y_values)*sum(a!=b for (a,b) in zip(classified,y_values))
print(error)


0.0032679738562091504
0.1764705882352941


## Dataset - Illumination

In [15]:
data = scipy.io.loadmat('Data/illumination.mat')
faces_flat = np.array(data['illum'])
#number of classes 
M = 68

### Split Data

In [16]:
train_idx = list(range(0,15))
test_idx = list(range(15,21))

# Vectorized selection
# training_faces: (pixels, 9 images * 68 subjects) first 9 images of subj 0, and so on
training_faces = faces_flat[:, train_idx, :].swapaxes(1, 2).reshape(faces_flat.shape[0], -1)

# test_faces: (pixels, 4 images * 68 subjects)
test_faces = faces_flat[:, test_idx, :].swapaxes(1, 2).reshape(faces_flat.shape[0], -1)

print("Training faces shape:", training_faces.shape)
print("Test faces shape:", test_faces.shape)

Training faces shape: (1920, 1020)
Test faces shape: (1920, 408)


### PCA and MDA

In [17]:
#PCA - reduce dimensions from 504 to 100 (only 400 obsercations, sigma_w will be non invertible for 504).
A = PCA(training_faces, 300)
X_PCA = A @ training_faces
X_test_PCA = A @ test_faces
print(X_PCA.shape)


#MDA - reduce dimensions from 100 to 50
A,class_means,sigma_w = MDA(X_PCA, 40, M)
A = np.real(A)
X_MDA = A @ X_PCA
X_test_MDA = A @ X_test_PCA
print(X_MDA.shape)

(300, 1020)
(40, 1020)


### Estimation

In [18]:
#after MDA, assumtion is all classes share covariance A@sigma_w@A.T, with means A@class_means
print(class_means.shape)
class_means = A @ class_means
sigma_w = A @ sigma_w @ A.T
print(sigma_w.shape)

(300, 68)
(40, 40)


### Bayes Classifier

In [19]:
# For shared covariance and equal priors, the classifier is the minimum Mahalanobis distance classifier
# training error
N_train = 1020
sigma_inv = np.linalg.inv(sigma_w)
#find mahalanobis distances for training data
distances = np.zeros((N_train,M))
for i in range(M):
    delta = X_MDA - class_means[:,i].reshape(-1,1)
    #vectorized
    sigma_delta = sigma_inv @ delta
    distances[:,i] = np.sum(delta * sigma_delta, axis = 0)
classified = np.argmin(distances, axis = 1)
y_values = [i for i in range(68) for _ in range(15)]

error = 1/len(y_values)*sum(a!=b for (a,b) in zip(classified,y_values))
print(error)

N_test = 408
#find mahalanobis distances for training data
distances = np.zeros((N_test,M))
for i in range(M):
    delta = X_test_MDA - class_means[:,i].reshape(-1,1)
    #vectorized
    sigma_delta = sigma_inv @ delta
    distances[:,i] = np.sum(delta * sigma_delta, axis = 0)
classified = np.argmin(distances, axis = 1)
y_values = [i for i in range(68) for _ in range(6)]

error = 1/len(y_values)*sum(a!=b for (a,b) in zip(classified,y_values))
print(error)


0.0
0.0
